<a href="https://colab.research.google.com/github/Newbluewood/ML-SentimentModelForReviews/blob/main/notebooks/product_category_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📥 Učitavanje i pregled skupa podataka

Pre analize učitavamo skup i gledamo strukturu — bez ikakvih izmena.

U ovom koraku:
- učitavamo CSV sa GitHub-a
- proveravamo broj redova i kolona
- prikazujemo prvih nekoliko redova

**Cilj projekta:** na osnovu **Product Title** predvideti **Category Label**.


In [ ]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/Newbluewood/"
    "ML-SentimentModelForReviews/main/data/IMLP4_TASK_03-products.csv"
)
df = pd.read_csv(url)
df.columns = df.columns.str.strip()


In [ ]:
print("Oblik skupa (redovi, kolone):", df.shape)
display(df.head())


**Zaključak:** Skup je učitan. Sledeći korak — tipovi kolona i broj popunjenih vrednosti.


In [ ]:
df.info()


## 🔍 Provera nedostajućih vrednosti

Nedostajući podaci kvare treniranje. Ovde samo **dijagnostikujemo** — još ne brišemo.


In [ ]:
print("Nedostajuće vrednosti po kolonama:")
print(df.isna().sum())


**Šta gledamo:** koje kolone imaju praznine i koliko ih ima.

Posebno su nam bitni **Product Title** i **Category Label** — bez njih red ne može u model.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, cmap="YlOrRd")
plt.title("Heatmap nedostajućih vrednosti")
plt.show()


**Zaključak:** Vizuelno vidimo gde su rupe. Pre čišćenja pregledamo raspodelu kategorija i ključne kolone.


## 📊 Analiza raspodele kategorija

Pregledamo ciljnu promenljivu na **sirovom** skupu — kao sentiment analiza u ITA-ML.


In [ ]:
raspodela_kategorija = df["Category Label"].value_counts()
print("Broj različitih kategorija:", df["Category Label"].nunique())
print(raspodela_kategorija)


**Šta tražimo:**
- da li su klase neuravnotežene
- da li postoje dupli nazivi iste kategorije (npr. `fridge` i `Fridges`)

Ako vidimo nekonzistentnosti, ispravićemo ih posle `dropna()`.


In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(
    data=df, y="Category Label",
    order=raspodela_kategorija.index,
    hue="Category Label", legend=False
)
plt.title("Raspodela proizvoda po kategorijama")
plt.tight_layout()
plt.show()


## 📝 Istraživanje kolone `Product Title`

Glavni ulaz modela. Pre čišćenja proveravamo kako je kolona zapisana.


In [ ]:
print("Tip kolone:", df["Product Title"].dtype)
print("\nPrvih 10 naslova:")
print(df["Product Title"].head(10).to_string(index=False))


In [ ]:
duzina_raw = df["Product Title"].astype(str).str.len()
print("Statistika dužine naslova (znakovi):")
print(duzina_raw.describe())
print("\nRedova bez naslova:", df["Product Title"].isna().sum())


**Zaključak:** Naslovi su tekst. Dužina varira po proizvodu — kasnije ćemo od toga napraviti karakteristiku.


## 💸 Istraživanje numeričkih kolona

Pregledamo `Merchant Rating` i `Number_of_Views` pre čišćenja — da znamo tipove i opseg vrednosti.


In [ ]:
print("Merchant Rating")
print("  tip:", df["Merchant Rating"].dtype)
print(df["Merchant Rating"].describe())
print("\nNumber_of_Views")
print("  tip:", df["Number_of_Views"].dtype)
print(df["Number_of_Views"].describe())


**Zaključak:** Obe kolone su numeričke. Zadržavamo ih do `dropna()`; detaljnu analizu korelacije sa kategorijom (kvartili + boxplot) radimo posle standardizacije kategorija.


## 🧹 Uklanjanje nedostajućih vrednosti

Sada uklanjamo **sve redove** gde bilo koja kolona ima `NaN` — isto kao `df.dropna()` u ITA-ML.

**Zašto?** Dobijamo potpuno popunjen skup bez imputacije.


In [ ]:
prethodni_broj = len(df)
df = df.dropna()
print("Pre:", prethodni_broj, "redova")
print("Posle:", df.shape[0], "redova")
print("Uklonjeno:", prethodni_broj - df.shape[0])


In [ ]:
print("Nedostajuće posle dropna:")
print(df.isna().sum())


**Zaključak:** Svaka ćelija ima vrednost. Sledeće — usklađivanje naziva kategorija.


## ✅ Standardizacija `Category Label`

Ispravljamo nekonzistentne nazive i postavljamo tip `category`.


In [ ]:
print("PRE standardizacije:")
print(df["Category Label"].value_counts())


In [ ]:
ispravka_kategorija = {
    "fridge": "Fridges",
    "CPU": "CPUs",
    "Mobile Phone": "Mobile Phones",
}
df["Category Label"] = (
    df["Category Label"].astype(str).str.strip().replace(ispravka_kategorija)
)
df["Category Label"] = df["Category Label"].astype("category")
print("POSLE standardizacije:")
print(df["Category Label"].value_counts())


**Zaključak:** Ostaje 10 čistih kategorija. Sledeće — čišćenje naslova.


## ✅ Uklanjanje razmaka u `Product Title`

Prvi korak čišćenja teksta — `strip()` sa početka i kraja.


In [ ]:
print("PRE:")
print(df["Product Title"].head(3).to_string(index=False))
df["Product Title"] = df["Product Title"].astype(str).str.strip()
print("\nPOSLE:")
print(df["Product Title"].head(3).to_string(index=False))
